# Label study: merged crowns against HOT's labels

Run on a GPU runtime. Set `REPO_SHA` in the next cell, then run the cells in order: setup, HOT data, predict, val sweep, test (once), sample views, label audit, summary.

The release model `NEON.pt` predicts crown boxes on every HOT validation and test unit. The merge step turns touching crowns into `tree_group` boxes and keeps the rest as `tree` boxes. Its settings (score threshold, gap, minimum crowns N) are tuned on HOT val and scored once on HOT test. The audit profiles how each training source labels dense canopy.

The predictions are cached per unit on Drive (`fair_trees/label_study/cache`): re-running the predict cell after a disconnect resumes. Results are written to `experiments/results/label-study` in the clone; download that folder at the end to commit it.

The repo is private: add a read-only GitHub token (Contents: read) under Colab Secrets (the key icon) as `GITHUB_TOKEN`, with notebook access on.

In [ ]:
REPO_SHA = ""
SEED = 0
MAX_REGIONS = 300  # regions per source in the label audit, sampled with SEED
# audit sources at pinned commits: restor/tcd main, and the CanopyRS gpkg branches (region-level files)
TCD_REVISION = "d97d4da0ebbb6e249ae95ac5e19656babd972eb2"
QUEBEC_REVISION = "16f3574ff8cfe798cf4f0c93b2f70d12468225f0"
SELVABOX_REVISION = "1f78a7a7fdcb27257df1f88efbf0b385c90c3dc9"

In [ ]:
import subprocess
import sys
from pathlib import Path

from google.colab import drive, userdata

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/fair_trees")

assert REPO_SHA, "set REPO_SHA to the pushed commit"
# the repo is private: a read-only GitHub token in Colab Secrets (key icon), named GITHUB_TOKEN
REPO = "github.com/OmdenaAI/HOTOSM-fAIr-Trees.git"
token = userdata.get("GITHUB_TOKEN")
!rm -rf /content/fair_trees
# git clone through subprocess so the token never reaches the cell output
url = f"https://x-access-token:{token}@{REPO}"
clone = subprocess.run(["git", "clone", "-q", url, "/content/fair_trees"])
# checked by hand: a CalledProcessError message would print the URL, token included
if clone.returncode:
    raise RuntimeError(f"git clone failed (exit {clone.returncode}): check the GITHUB_TOKEN secret")
# drop the token from the saved remote so it is not left in .git/config
git = ["git", "-C", "/content/fair_trees"]
subprocess.run([*git, "remote", "set-url", "origin", f"https://{REPO}"], check=True)
subprocess.run([*git, "checkout", "-q", REPO_SHA], check=True)
del token, url

%pip install "/content/fair_trees[train,test]" -r /content/fair_trees/experiments/requirements.txt
%cd /content/fair_trees
sys.path.insert(0, "/content/fair_trees")

repo = Path("/content/fair_trees")
CACHE = DRIVE / "label_study" / "cache"  # one record per unit on Drive, so a disconnect loses nothing
RESULTS = repo / "experiments/results/label-study"
RESULTS.mkdir(parents=True, exist_ok=True)

## HOT data

Labels and tile footprints for every split, imagery for validation and test. The masking cost is what masking `tree_group` polygons out of training would remove from HOT train.

In [ ]:
import json

import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download

from experiments.label_study.hot import REPO_ID, REVISION, masking_cost, read_hot

SPLITS = ("validation", "test")  # as named in tiles.parquet


def fetch(path):
    return hf_hub_download(REPO_ID, path, repo_type="dataset", revision=REVISION)


data = read_hot(fetch("stac/project-50661/labels.parquet"), fetch("stac/project-50661/tiles.parquet"))
images = {}  # tile_id -> encoded image
for split in SPLITS:
    table = pq.read_table(fetch(f"data/{split}-00000-of-00001.parquet"), columns=["tile_id", "image"])
    encoded = (v["bytes"] for v in table["image"].to_pylist())
    images.update(zip(table["tile_id"].to_pylist(), encoded, strict=True))
print(f"{len(data.trees):,} trees, {len(data.groups):,} groups, {len(images):,} val and test tiles")

cost = masking_cost(data, "train")
(RESULTS / "masking_cost.json").write_text(json.dumps(cost, indent=2))
print(json.dumps(cost, indent=2))

## Predict

One mosaic per unit (a connected run of tiles in a 16 × 16 tile block), serving defaults. Re-running the cell resumes: units with a record in the cache are skipped.

In [ ]:
import shutil

import torch

from experiments.label_study.hot import split_units, unit_mosaic, unit_truth
from experiments.label_study.study import run_units
from fair_trees.params import InferenceParams
from fair_trees.serve import predict_mosaic
from fair_trees.train import build_model, torch_run_fn

SCRATCH = Path("/content/label_study_units")  # PNG chips of the unit being built
run_fn = torch_run_fn(build_model(DRIVE / "weights" / "NEON.pt", trainable_layers=0), torch.device("cuda"))


def predict(mosaic):
    return predict_mosaic(run_fn, mosaic, InferenceParams())


def unit_builder(tile_ids, workdir):
    def build():
        shutil.rmtree(workdir, ignore_errors=True)  # left over from an interrupted run
        mosaic = unit_mosaic(images, tile_ids, workdir)
        return mosaic, unit_truth(data, mosaic, tile_ids)

    return build


units = [
    (unit_id, split, unit_builder(tile_ids, SCRATCH / unit_id))
    for split in SPLITS
    for unit_id, tile_ids in split_units(data.tiles[data.tiles["split"] == split], split)
]
builders = {unit_id: build for unit_id, _, build in units}
predicted = run_units(predict, units, CACHE)
done = sum((CACHE / f"{unit_id}.npz").exists() for unit_id in builders)
print(f"predicted {predicted} units now; {done} / {len(units)} units done")

## Val sweep

Every (score, gap, N) setting on HOT val; the chosen one has the highest mean of tree F1 and group F1 (IoU 0.4). The plot shows the trade-off in N at the chosen score and gap.

In [ ]:
import matplotlib.pyplot as plt

from experiments.label_study.study import choose, load_records, sweep

table = sweep(load_records(CACHE, "validation"))
table.to_csv(RESULTS / "sweep_val.csv", index=False)
setting = choose(table)
print(table.sort_values("mean_f1", ascending=False, kind="stable").head(10).to_string(index=False))
print("chosen:", setting)

curve = table[(table["score"] == setting["score"]) & (table["gap_m"] == setting["gap_m"])]
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(curve["min_crowns"], curve["tree_f1"], marker="o", label="tree F1")
ax.plot(curve["min_crowns"], curve["group_f1"], marker="o", label="group F1 (IoU 0.4)")
ax.axvline(setting["min_crowns"], color="grey", linestyle=":")
ax.set(xlabel="N (min crowns per group)", ylabel="F1 on HOT val")
ax.set_title(f"score > {setting['score']}, gap {setting['gap_m']} m")
ax.legend()
fig.savefig(RESULTS / "sweep_val.png", dpi=120, bbox_inches="tight")
plt.show()

## Test, once

The chosen setting on HOT test. Run this cell once, after the sweep; do not tune on its output.

In [ ]:
from experiments.label_study.study import score_test

test = score_test(load_records(CACHE, "test"), setting)
(RESULTS / "test.json").write_text(json.dumps(test, indent=2))
for subset in ("all", "dense"):
    print(f"{subset} ({test[subset]['units']} units)")
    for key in ("trees", "groups_40", "groups_25"):
        m = test[subset][key]
        lo, hi = m["f1_ci"]
        print(f"  {key:9} P {m['precision']:.3f}  R {m['recall']:.3f}  F1 {m['f1']:.3f} [{lo:.3f}, {hi:.3f}]")
print("predicted tree box side / HOT crown diameter:", test["diameter_ratio"])

## Sample views

Four test units (two dense, two not): predicted `tree` boxes (yellow) and `tree_group` boxes (magenta), HOT tree points (red) and HOT group boxes (cyan, dashed). Then one unit's unmatched tree boxes, to judge by eye whether they are trees HOT did not map.

In [ ]:
import random

import numpy as np
from matplotlib.patches import Rectangle

from experiments.label_study.merge import merge_crowns
from experiments.label_study.score import match_points
from fair_trees.decode import Detections

test_records = load_records(CACHE, "test")
rng = random.Random(SEED)
dense = [r for r in test_records if r.dense]
views = rng.sample(dense, 2) + rng.sample([r for r in test_records if not r.dense], 2)


def merged_at_setting(rec):
    """The chosen setting's merge, and which of its tree boxes have their centre in a HOT group."""
    keep = rec.scores > setting["score"]
    dets = Detections(rec.boxes[keep], rec.scores[keep])
    merged = merge_crowns(dets, setting["gap_m"] / rec.gsd_m, setting["min_crowns"])
    return merged, rec.in_group[keep][merged.tree_index]


def draw_boxes(ax, boxes, color, **style):
    style = {"linewidth": 1} | style
    for x0, y0, x1, y1 in boxes:
        ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor=color, **style))


fig, axes = plt.subplots(2, 2, figsize=(14, 14))
for ax, rec in zip(axes.flat, views, strict=True):
    merged, _ = merged_at_setting(rec)
    ax.imshow(builders[rec.unit_id]()[0].rgb)
    draw_boxes(ax, merged.trees.boxes, "yellow")
    draw_boxes(ax, merged.groups.boxes, "magenta", linewidth=2)
    draw_boxes(ax, rec.group_boxes, "cyan", linestyle="--")
    ax.scatter(*rec.tree_points.T, s=8, c="red")
    ax.set_title(f"{rec.unit_id} ({'dense' if rec.dense else 'not dense'})")
    ax.axis("off")
fig.savefig(RESULTS / "sample_views.png", dpi=100, bbox_inches="tight")
plt.show()

rec = views[2]  # not dense: most of its HOT trees are points
merged, in_group = merged_at_setting(rec)
trees = merged.trees.boxes[~in_group]  # the boxes score_test counts
matched = [i for i, _ in match_points(trees, rec.tree_points)]
unmatched = trees[np.isin(np.arange(len(trees)), matched, invert=True)]
fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(builders[rec.unit_id]()[0].rgb)
draw_boxes(ax, unmatched, "yellow", linewidth=2)
ax.scatter(*rec.tree_points.T, s=8, c="red")
ax.set_title(f"{rec.unit_id}: {len(unmatched)} of {len(trees)} tree boxes hold no HOT point")
ax.axis("off")
fig.savefig(RESULTS / "unmatched_trees.png", dpi=100, bbox_inches="tight")
plt.show()

## Label audit

How each source labels dense canopy, clustered with the chosen gap and N. At most `MAX_REGIONS` regions per source:
- HOT: one region per split; trees are squares of their `diameter_crown`.
- OAM-TCD (`restor/tcd`, train and test): one region per 2048 px tile; category 2 is tree, 1 is canopy (group). Its 10 cm GSD is in EPSG:3395 units, so the ground GSD is 0.1 m × cos(latitude).
- NEON: the annotated training rasters of the raw archive on Drive, 10 cm.
- QuebecTrees and SelvaBox: the `gpkg` branch, one region per AOI file (a raster's train, valid or test area); its tiles on `main` overlap.

In [ ]:
import math
import zipfile

import geopandas as gpd
import pandas as pd
from huggingface_hub import HfApi, HfFileSystem

from experiments.converters.neon import find_training_rasters
from experiments.label_study.audit import audit_table, coco_labels, geo_labels, neon_labels
from experiments.label_study.hot import MAX_DIAMETER_M, hot_labels


def pick(items):
    """At most MAX_REGIONS of the items, sampled with SEED."""
    items = sorted(items)
    return random.Random(SEED).sample(items, min(MAX_REGIONS, len(items)))


def concat(parts):
    """One (labels, regions) pair from a list of them."""
    return tuple(pd.concat(frames, ignore_index=True) for frames in zip(*parts, strict=True))


def tcd_labels():
    fs = HfFileSystem()
    files = sorted(fs.glob(f"datasets/restor/tcd@{TCD_REVISION}/data/*.parquet"))
    meta = pd.concat(pq.read_table(f, columns=["image_id", "crs"], filesystem=fs).to_pandas() for f in files)
    assert (meta["crs"] == "EPSG:3395").all(), "the GSD below assumes EPSG:3395 tiles"
    picked = pick(meta["image_id"].tolist())
    columns = ["image_id", "width", "height", "lon", "coco_annotations"]
    parts = []
    for f in files:
        table = pq.read_table(f, columns=columns, filesystem=fs, filters=[("image_id", "in", picked)])
        for row in table.to_pylist():
            image = {"id": row["image_id"], "width": row["width"], "height": row["height"]}
            coco = {"images": [image], "annotations": json.loads(row["coco_annotations"])}
            # 0.1 m is in EPSG:3395 units; the parquet's lat and lon columns are swapped: lon is the latitude
            gsd_m = 0.1 * math.cos(math.radians(row["lon"]))
            parts.append(coco_labels(coco, source="oam-tcd", gsd_m=gsd_m, tree_ids={2}, group_ids={1}))
    return concat(parts)


def neon_training_labels():
    raw_drive = DRIVE / "raw" / "neon"
    listed = {f["name"] for f in json.loads((raw_drive / "manifest.json").read_text())["files"]}
    raw_root = Path("/content/raw/neon")
    shutil.rmtree(raw_root, ignore_errors=True)
    # annotations.zip carries its own top folder; training.zip has RGB/, CHM/, ... at its root
    extract = (("annotations.zip", "annotations/", raw_root), ("training.zip", "RGB/", raw_root / "training"))
    for name, prefix, target in extract:
        assert name in listed, f"{name} is not in the raw manifest on Drive"
        with zipfile.ZipFile(raw_drive / name) as z:
            z.extractall(target, members=[m for m in z.namelist() if m.startswith(prefix)])
    return neon_labels(pick(find_training_rasters(raw_root)))


def gpkg_labels(repo_id, revision, source):
    """A CanopyRS gpkg-branch source, one region per AOI file; boxes belong to the AOI under their centre."""

    def get(name):
        return hf_hub_download(repo_id, name, repo_type="dataset", revision=revision)

    files = HfApi().list_repo_files(repo_id, repo_type="dataset", revision=revision)
    aois = pick(f for f in files if "_aoi_" in f)
    parts = []
    for stem in sorted({f.split("_aoi_")[0] for f in aois}):  # <raster>_labels_<masks|boxes>
        crowns = gpd.read_file(get(f"{stem}.gpkg"))
        utm = crowns.estimate_utm_crs()
        crowns = crowns.to_crs(utm)
        mine = [f for f in aois if f.startswith(f"{stem}_aoi_")]
        regions = pd.concat(gpd.read_file(get(f)).assign(region=f.removesuffix(".gpkg")) for f in mine)
        regions = regions[["region", "geometry"]].dissolve("region").reset_index().to_crs(utm)
        centres = gpd.GeoDataFrame(geometry=crowns.representative_point(), crs=utm)
        under = centres.sjoin(regions, predicate="within")
        under = under[~under.index.duplicated()]  # a centre on a shared AOI edge: first
        crowns = crowns.loc[under.index].assign(region=under["region"])
        parts.append(geo_labels(crowns, regions, source=source))
    return concat(parts)


parts = {
    "hot": hot_labels(data),
    "oam-tcd": tcd_labels(),
    "neon": neon_training_labels(),
    "quebectrees": gpkg_labels("CanopyRS/QuebecTrees", QUEBEC_REVISION, "quebectrees"),
    "selvabox": gpkg_labels("CanopyRS/SelvaBox", SELVABOX_REVISION, "selvabox"),
}
audit = audit_table(parts, gap_m=setting["gap_m"], min_crowns=setting["min_crowns"])
audit.to_csv(RESULTS / "audit.csv")
print(audit.T.to_string(float_format=lambda v: f"{v:.3g}"))
hot_trees = parts["hot"][0].query("cls == 'tree'")
print(
    "HOT: crowns_per_ha counts all trees; crown sizes and the touching and HOT-rule shares use only the"
    f" {hot_trees['w_m'].notna().mean():.0%} with a parseable diameter_crown in (0, {MAX_DIAMETER_M:.0f}] m."
)

## Summary

In [ ]:
def f1_with_ci(m):
    lo, hi = m["f1_ci"]
    return f"{m['f1']:.3f} [{lo:.3f}, {hi:.3f}]"


ratio = test["diameter_ratio"]
summary = {
    "chosen on val": f"score > {setting['score']}, gap {setting['gap_m']} m, N {setting['min_crowns']}",
    "test tree F1, all": f1_with_ci(test["all"]["trees"]),
    "test tree F1, dense": f1_with_ci(test["dense"]["trees"]),
    "test group F1 @0.4, all": f1_with_ci(test["all"]["groups_40"]),
    "test group F1 @0.4, dense": f1_with_ci(test["dense"]["groups_40"]),
    "box / HOT diameter, median [IQR]": f"{ratio['median']:.2f} [{ratio['q25']:.2f}, {ratio['q75']:.2f}]",
    "HOT train area inside groups": f"{cost['group_area_share']:.1%}",
    "HOT train trees inside groups": f"{cost['trees_in_groups']:,} of {cost['trees_total']:,}",
}
for source, row in audit.iterrows():
    summary[f"{source}: hot_rule_group_share"] = f"{row['hot_rule_group_share']:.3f}"
    summary[f"{source}: groups_over_40m"] = f"{row['groups_over_40m']:.3f}"
    summary[f"{source}: group_crowns_equiv_median"] = f"{row['group_crowns_equiv_median']:.1f}"
print(pd.Series(summary).to_string())

drive.flush_and_unmount()